# NB24: DuckDB & Engine Abstraction

siege_utilities provides an **engine-agnostic DataFrame API** that lets you write
analysis code once and run it on different backends:

- **pandas** — default, great for small-medium data
- **DuckDB** — columnar analytics engine, 10-100x faster for SQL and aggregations
- **Spark** — distributed, for data that doesn't fit on one machine
- **PostGIS** — spatial queries directly in the database

This notebook shows how to switch engines and when each one shines.

In [ ]:
import time
import numpy as np
import pandas as pd

from siege_utilities.data.dataframe_engine import (
    Engine,
    get_engine,
    PandasEngine,
    DuckDBEngine,
)

print("Available engines:", [e.value for e in Engine])

## 1. Engine Factory

In [ ]:
# Get engine by name or enum
pandas_engine = get_engine("pandas")
print(f"Engine: {pandas_engine.name}")

duckdb_engine = get_engine(Engine.DUCKDB)
print(f"Engine: {duckdb_engine.name}")

## 2. Generate Test Data

In [ ]:
# Generate a realistic dataset: 100K contribution records
np.random.seed(42)
n = 100_000

states = np.random.choice(["CA", "TX", "NY", "FL", "PA", "OH", "IL", "GA", "NC", "MI"], n)
parties = np.random.choice(["DEM", "REP", "IND"], n, p=[0.45, 0.45, 0.10])
amounts = np.random.lognormal(mean=4.0, sigma=1.5, size=n).round(2)
years = np.random.choice([2018, 2020, 2022, 2024], n)

df = pd.DataFrame({
    "state": states,
    "party": parties,
    "amount": amounts,
    "year": years,
    "donor_id": np.random.randint(1, 20000, n),
})
print(f"Test data: {len(df):,} rows, {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

## 3. Engine-Agnostic Operations

The same code works with any engine — just swap the engine parameter.

In [ ]:
def run_analysis(engine, df):
    """Run a standard analysis pipeline on any engine."""
    t0 = time.perf_counter()
    
    # Group by state and party, aggregate
    result = engine.groupby_agg(
        df,
        group_cols=["state", "party"],
        agg_dict={"amount": "sum", "donor_id": "count"},
    )
    
    # Convert to pandas for display
    result_pd = engine.to_pandas(result)
    
    elapsed = time.perf_counter() - t0
    return result_pd, elapsed

# Run on pandas
result_pd, time_pd = run_analysis(pandas_engine, df)
print(f"Pandas: {time_pd:.4f}s, {len(result_pd)} rows")

# Run on DuckDB
result_duck, time_duck = run_analysis(duckdb_engine, df)
print(f"DuckDB: {time_duck:.4f}s, {len(result_duck)} rows")

print(f"\nSpeedup: {time_pd / time_duck:.1f}x")

## 4. SQL Queries — DuckDB's Sweet Spot

DuckDB shines when you want SQL against in-memory data.

In [ ]:
# DuckDB can query pandas DataFrames directly with SQL
result = duckdb_engine.query(
    """
    SELECT 
        state,
        party,
        COUNT(*) as num_contributions,
        ROUND(SUM(amount), 2) as total_amount,
        ROUND(AVG(amount), 2) as avg_amount,
        ROUND(MEDIAN(amount), 2) as median_amount
    FROM df
    WHERE year = 2024
    GROUP BY state, party
    ORDER BY total_amount DESC
    LIMIT 10
    """,
    df=df,
)
print(duckdb_engine.to_pandas(result).to_string(index=False))

## 5. Joins

In [ ]:
# Create a state metadata table
state_meta = pd.DataFrame({
    "state": ["CA", "TX", "NY", "FL", "PA", "OH", "IL", "GA", "NC", "MI"],
    "region": ["West", "South", "Northeast", "South", "Northeast",
               "Midwest", "Midwest", "South", "South", "Midwest"],
    "electoral_votes": [54, 40, 28, 30, 19, 17, 19, 16, 16, 15],
})

# Join with the engine
joined = duckdb_engine.join(df, state_meta, on="state", how="inner")
joined_pd = duckdb_engine.to_pandas(joined)
print(f"Joined: {len(joined_pd):,} rows")

# Regional analysis via SQL
regional = duckdb_engine.query("""
    SELECT region, ROUND(SUM(amount), 2) as total, COUNT(*) as n
    FROM df
    GROUP BY region
    ORDER BY total DESC
""", df=joined_pd)
print(duckdb_engine.to_pandas(regional).to_string(index=False))


## 6. When to Use Which Engine

| Engine | Best For | Data Size | Setup |
|--------|----------|-----------|-------|
| **pandas** | Quick exploration, plotting, small data | < 1M rows | Built-in |
| **DuckDB** | SQL queries, aggregations, analytical workloads | 1M - 1B rows | `pip install duckdb` |
| **Spark** | Distributed processing, cluster workloads | > 1B rows | Cluster required |
| **PostGIS** | Spatial joins, geometry operations | Any | PostgreSQL + PostGIS |

**Rule of thumb:** Start with pandas. If queries are slow, switch to DuckDB.
If data doesn't fit in memory, use Spark.